f > f bag > z

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from data.dataset import TCGABags
from models.network import MonoCMIL
from core.losses import HypersphereLoss, Phase2Loss
from core.memory import FeatureMemoryBank

base_path = '/data3/jinsol/TCGA/features_conch_256/pt_files'
incremental_classes = ['TCGA-BRCA', 'TCGA-LUAD', 'TCGA-COAD']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MonoCMIL(input_dim=512, hidden_dim=512, z_dim=256).to(device)

criterion_phase1 = HypersphereLoss(reduction='mean').to(device)
criterion_phase2 = Phase2Loss().to(device)
memory_bank = FeatureMemoryBank(z_dim=256)

optimizer_phase1 = optim.Adam(model.abmil.parameters(), lr=0.001)
optimizer_phase2 = optim.Adam(model.mlp_head.parameters(), lr=0.001)

print("🔥 MonoCMIL 순차 학습 검증 파이프라인 시작...\n")

for task_id, cls_name in enumerate(incremental_classes):
    print("=" * 80)
    print(f"🚀 [Task {task_id}] '{cls_name}' 학습 시작")
    print("=" * 80)
    
    cls_files = glob.glob(os.path.join(base_path, f'{cls_name}/*.pt'))[:2]
    data_list = [(f, task_id) for f in cls_files]
    dataloader = DataLoader(TCGABags(data_list), batch_size=1, shuffle=True)
    
    # --- Phase 1 ---
    model.train()
    optimizer_phase1.zero_grad()
    z_list, f_bag_list = [], []
    
    for features, label in dataloader:
        features = features.to(device)
        z, f_bag, A = model(features)
        z_list.append(z)
        f_bag_list.append(f_bag)
        
    batch_z = torch.cat(z_list, dim=0)
    batch_labels = torch.full((batch_z.shape[0],), task_id, dtype=torch.long).to(device)
    
    loss_p1, _ = criterion_phase1(batch_z, batch_labels)
    loss_p1.backward()
    optimizer_phase1.step()
    
    print(f"✅ [Phase 1] Hypersphere Loss: {loss_p1.item():.4f}")
    
    # --- Memory Update & Anchor Generation ---
    batch_f_bag = torch.cat(f_bag_list, dim=0).detach()
    memory_bank.update_statistics(task_id, batch_f_bag)
    current_anchor = memory_bank.generate_orthogonal_anchor(task_id, device)
    
    if task_id > 0:
        sims = [F.cosine_similarity(current_anchor.unsqueeze(0), a.unsqueeze(0)).item() for k, a in memory_bank.anchors.items() if k != task_id]
        print(f"✅ [Memory] 기존 앵커들과의 코사인 유사도(직교성 검증): {[round(s, 4) for s in sims]}")
    else:
        print(f"✅ [Memory] 첫 앵커 생성 완료")

    # --- Phase 2 ---
    optimizer_phase2.zero_grad()
    z_cur = model.mlp_head(batch_f_bag)
    f_past_dict = memory_bank.sample_past_features(task_id, batch_size=batch_z.shape[0], f_cur=batch_f_bag)
    
    z_past_dict = {}
    for past_id, f_past in f_past_dict.items():
        z_past_dict[past_id] = model.mlp_head(f_past.to(device))
        
    loss_p2 = criterion_phase2(z_cur, z_past_dict, current_anchor)
    loss_p2.backward()
    optimizer_phase2.step()
    
    print(f"✅ [Phase 2] Directional Alignment Loss: {loss_p2.item():.4f}")
    
    # --- 학습 결과 검증 ---
    with torch.no_grad():
        sim_cur = F.cosine_similarity(model.mlp_head(batch_f_bag), current_anchor.unsqueeze(0)).mean().item()
        print(f"   📊 [검증] 현재 데이터 <-> 현재 앵커 유사도 (높을수록 좋음): {sim_cur:.4f}")
        
        if task_id > 0:
            for past_id, f_past in f_past_dict.items():
                z_past_updated = model.mlp_head(f_past.to(device))
                sim_past = F.cosine_similarity(z_past_updated, current_anchor.unsqueeze(0)).mean().item()
                print(f"   📊 [검증] 과거 Task {past_id} 가짜 데이터 <-> 현재 앵커 유사도 (낮을수록 좋음): {sim_past:.4f}")
    print("\n")

🔥 MonoCMIL 순차 학습 검증 파이프라인 시작...

🚀 [Task 0] 'TCGA-BRCA' 학습 시작
✅ [Phase 1] Hypersphere Loss: 0.0937
✅ [Memory] 첫 앵커 생성 완료
✅ [Phase 2] Directional Alignment Loss: 0.6709
   📊 [검증] 현재 데이터 <-> 현재 앵커 유사도 (높을수록 좋음): 0.5253


🚀 [Task 1] 'TCGA-LUAD' 학습 시작
✅ [Phase 1] Hypersphere Loss: 0.0585
✅ [Memory] 기존 앵커들과의 코사인 유사도(직교성 검증): [-0.0872]
✅ [Phase 2] Directional Alignment Loss: 0.6801
   📊 [검증] 현재 데이터 <-> 현재 앵커 유사도 (높을수록 좋음): 0.1536
   📊 [검증] 과거 Task 0 가짜 데이터 <-> 현재 앵커 유사도 (낮을수록 좋음): -0.0822


🚀 [Task 2] 'TCGA-COAD' 학습 시작
✅ [Phase 1] Hypersphere Loss: 0.0754
✅ [Memory] 기존 앵커들과의 코사인 유사도(직교성 검증): [0.0404, 0.0328]
✅ [Phase 2] Directional Alignment Loss: 0.7072
   📊 [검증] 현재 데이터 <-> 현재 앵커 유사도 (높을수록 좋음): 0.0531
   📊 [검증] 과거 Task 0 가짜 데이터 <-> 현재 앵커 유사도 (낮을수록 좋음): -0.0397
   📊 [검증] 과거 Task 1 가짜 데이터 <-> 현재 앵커 유사도 (낮을수록 좋음): -0.0275


